# Retrieval-Augmented Generation (RAG) Course - Explained (موضح باللغة العربية)

هذا الملف يجمع كافة الدروس العملية والتمارين لتطبيقات التوليد المعزز بالاسترجاع (RAG) وقواعد البيانات المتجهة موضحاً بالتفصيل باللغة العربية خطوة بخطوة.


## الدرس الأول: تقسيم المستندات إلى مقاطع (Document Chunking)


### أولاً: تعريف الدالة وتقسيم النص (Define chunking function)

نقوم بتحديد نص المستند وبناء دالة تقسم النص لشرائح بطول محدد مع تداخل طفيف لحفظ السياق اللغوي المتصل.


In [ ]:
# Step 1) تعريف النص وتقسيمه / Define raw text and chunk_text function
document = """
Retrieval-Augmented Generation (RAG) is a technique that grants LLMs access to external data.
RAG combines retrieval of documents with generation of text.
First, we load the knowledge source and split it into smaller text chunks.
Chunking is necessary because models have limited context windows.
We usually use overlapping chunks to ensure semantic continuity between borders.
"""

def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
    return chunks


### ثانياً: تجربة وظيفة التقسيم وعرض المقاطع (Test chunking output)

نقوم بتشغيل الدالة وعرض المقاطع الناتجة لرؤية التداخل اللغوي بين نهاية وبداية كل مقطع.


In [ ]:
# Step 2) تجربة التقسيم بمقاييس مختلفة / Test chunking function
chunks = chunk_text(document, chunk_size=120, overlap=30)
print(f"Total Chunks Generated: {len(chunks)}")
print("="*50)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}:\n{c.strip()}\n")


## الدرس الثاني: ترميز النصوص والبحث الجيبي (Vector Search)


### أولاً: استدعاء نموذج المتجهات وتوليدها (Generate Vector Embeddings)

نستدعي نموذج متجهات خفيف ومدرّب مسبقاً لتحويل مقاطع النصوص الأربعة لمصفوفة متجهات رقمية.


In [ ]:
# Step 1) استيراد المكتبات وتوليد المتجهات / Import embedding library and generate vectors
# %pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
chunks = [
    "Retrieval-Augmented Generation (RAG) grants LLMs access to external data.",
    "RAG combines document retrieval with text generation.",
    "Chunking splits text into smaller pieces because of context limits.",
    "Overlapping chunks ensure semantic continuity between borders."
]

chunk_embeddings = model.encode(chunks)
print("Embedding Matrix Shape:", chunk_embeddings.shape)


### ثانياً: حساب التشابه الجيبي للاستعلام (Calculate similarity score)

نقوم بتمثيل دالة التشابه الجيبي يدوياً لمقارنة متجه سؤال المستخدم بمتجهات كافة المقاطع المخزنة.


In [ ]:
# Step 2) حساب درجة التشابه الجيبي / Calculate Cosine Similarity manual function
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "How does RAG access external databases?"
query_embedding = model.encode(query)

scores = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]
print(f"Query: '{query}'")


### ثالثاً: ترتيب نتائج البحث الدلالي (Rank search results)

نرتب المقاطع من الأكثر دلالة وتشابهاً (الأقرب لـ 1.0) إلى الأقل دلالة وعرضها للمستخدم.


In [ ]:
# Step 3) ترتيب وتصفية نتائج البحث / Rank search results
ranked_indices = np.argsort(scores)[::-1]
print("Ranked Search Results:")
print("="*50)
for idx in ranked_indices:
    print(f"Score: {scores[idx]:.4f} | Chunk: '{chunks[idx]}'")


## الدرس الثالث: نظام التوليد المسترجع الكامل (End-to-End RAG Pipeline)


### أولاً: استدعاء نماذج التوليد والبحث وتحضير قاعدة المعرفة (Setup models & Database)

نقوم بتحميل نموذج المتجهات ونموذج التوليد GPT-2 وتجهيز قاعدة بيانات المعرفة التي سيبحث بها النظام.


In [ ]:
# Step 1) استدعاء النماذج وتجهيز قاعدة المعرفة / Load models and setup knowledge base
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
generator = pipeline("text-generation", model="gpt2")

knowledge_base = [
    "The capital of France is Paris. It is known for Eiffel Tower.",
    "The capital of Japan is Tokyo. It is famous for its sushi and technology.",
    "The capital of Australia is Canberra. It was selected as a compromise between Sydney and Melbourne."
]
kb_embeddings = embed_model.encode(knowledge_base)


### ثانياً: بناء دالة الاسترجاع الدلالي (Define retrieval function)

نقوم ببناء دالة تأخذ سؤال المستخدم وتبحث في قاعدة المعرفة لترجع المقطع الأكثر صلة وإفادة للاستعلام.


In [ ]:
# Step 2) بناء دالة الاسترجاع الدلالي / Define retrieval function
def retrieve(query):
    q_emb = embed_model.encode(query)
    scores = [np.dot(q_emb, kb_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(kb_emb)) for kb_emb in kb_embeddings]
    best_idx = np.argmax(scores)
    return knowledge_base[best_idx]


### ثالثاً: دمج السياق وصياغة الموجه وتوليد الإجابة (RAG execution)

نقوم بدمج السياق المجلوب مع سؤال المستخدم في قالب موجه (Prompt) موحد، ونرسله لنموذج التوليد لصياغة الإجابة الموثقة بالسياق.


In [ ]:
# Step 3) دمج السياق وتوليد الإجابة / RAG Execution
query = "What is the capital of Japan and what is it famous for?"
context = retrieve(query)

# Construct Prompt
prompt = f"Answer the query based on the context.\nContext: {context}\nQuery: {query}\nAnswer:"

# Generate text
output = generator(prompt, max_new_tokens=20, pad_token_id=50256)
print("RAG Prompt:")
print(prompt)
print("\nGenerated Response:")
print(output[0]['generated_text'])
